# NB04 — Live policy export

**Question.** The live app opens **c_only** sessions only (mastery unknown). Can we train a c_only Q table with the NB02 settings, check it against the rule, and export both live tables
(`q_c_only`, `q_main`) with golden test vectors for the backend?

**Outputs.** `outputs/models/q_c_only.json`; `backend/provenance/c_policy/{q_c_only.json, q_main.json, MANIFEST.json, golden_vectors.json}`; `outputs/figures|tables/04_*`.

`rl_core.agent.state_of` gained an *unknown* mastery bucket (index 3) for `mastery is None`; the 135 combined states are unchanged (q_main.json is still valid). `run_learner` gained a
`session_mode` argument. In c_only there is no mastery, so the simulator additions (growth, boredom dropout) do not apply: dropout is 0 and there is no mastery gain.

**Disclosure.** Literature-informed synthetic data — not clinical evidence. Simulation-only; not approved for use with children.

In [1]:
import sys, json, time, hashlib
from pathlib import Path
T_NB = time.time()
RL_ROOT = Path.cwd().resolve().parent
for p in (RL_ROOT, RL_ROOT / "notebooks"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import rlconfig as cfg
import rl_core
from rl_core import env as E
from rl_core.env import SimConfig, run_learner, sample_learner, rule_policy
from rl_core.agent import QAgent, AgentPolicy, state_of, WEIGHTS, UNKNOWN_MASTERY

import numpy as np, pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)
for d in (cfg.TABLES_DIR, cfg.MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
save_table = lambda df, n: df.to_csv(cfg.TABLES_DIR / f"04_{n}.csv", index=False)
SIM = SimConfig(growth=True, boredom=True, eta=E.ETA)     # does nothing in c_only (combined-mode additions)
TAU = 0.70                                                # the NB02 value (derived from the simulator), unchanged
pm = lambda x: f"{x.mean():.4f} ± {x.std() / np.sqrt(x.count()):.4f}"
pdiff = lambda x: f"{x.mean():+.4f} ± {x.std() / np.sqrt(x.count()):.4f}"
print(cfg.DISCLOSURE, "| tau =", TAU)

Literature-informed synthetic data — not clinical evidence. Simulation-only; not approved for use with children. | tau = 0.7


## 1. Train the c_only Q table

Same settings as NB02: 3,000 learners × 25 sessions, eps 0.10, alpha 0.10, gamma 0.90, min_tasks 3, main weights, training seeds 1000+ (< 9,000,000), c_only mode.

In [2]:
TRAIN_SEED0, N_TRAIN, SESSIONS = 1000, 3000, 25
agent = QAgent(); pol = AgentPolicy(agent, TAU, WEIGHTS, seed=TRAIN_SEED0)
t0, blockr = time.time(), []
for i in range(N_TRAIN):
    pol.rewards.clear()
    run_learner(pol, sample_learner(TRAIN_SEED0 + i), n_sessions=SESSIONS, seed=TRAIN_SEED0 + i, sim=SIM, session_mode="c_only")
    blockr.append((np.sum(pol.rewards), len(pol.rewards)))
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{N_TRAIN}  {time.time() - t0:.0f}s  states visited {len(agent.visits)}", flush=True)
T_TRAIN = time.time() - t0
B = np.array(blockr)
print(f"trained {N_TRAIN} learners x {SESSIONS} sessions in {T_TRAIN / 60:.1f} min; {int(B[:, 1].sum()):,} learning decisions; states visited {len(agent.visits)} (all with unknown mastery: {all(s[3] == UNKNOWN_MASTERY for s in agent.visits)})")
print("mean reward / decision: first 500 learners %.4f, last 500 learners %.4f" % (B[:500, 0].sum() / B[:500, 1].sum(), B[-500:, 0].sum() / B[-500:, 1].sum()))
agent.save(cfg.MODELS_DIR / "q_c_only.json", tau=TAU, weights=WEIGHTS, n_train_learners=N_TRAIN, sessions_per_learner=SESSIONS, seed=TRAIN_SEED0,
           session_mode="c_only", disclosure=cfg.DISCLOSURE)
Q_C = QAgent.load(cfg.MODELS_DIR / "q_c_only.json"); Q_M = QAgent.load(cfg.MODELS_DIR / "q_main.json")

  500/3000  134s  states visited 18


  1000/3000  292s  states visited 18


  1500/3000  455s  states visited 18


  2000/3000  620s  states visited 18


  2500/3000  782s  states visited 18


  3000/3000  943s  states visited 18


trained 3000 learners x 25 sessions in 15.7 min; 749,928 learning decisions; states visited 18 (all with unknown mastery: True)
mean reward / decision: first 500 learners 0.5241, last 500 learners 0.5223


### Findings — step 1

- **c_only Q table trained:** 3,000 learners × 25 sessions (seeds 1000–3999), **15.7 min**, 749,928 learning decisions. Only **18 states** are ever visited: with unknown mastery and the c_only cap (levels 1–2) the state space collapses to difficulty {1, 2} × first-try {3} × strain {3} × unknown = 18.
- Mean reward per decision is flat from the start (first 500 learners 0.5241, last 500 learners 0.5223), as in NB02: 18 states converge inside the first block.
- Saved to `outputs/models/q_c_only.json`. The 135 combined states of `q_main.json` are unchanged by the new *unknown* bucket (index 3).

## 2. Quick check — greedy c_only Q vs rule

100 new learners (seeds 9,300,000+, different from NB02's and NB03's), 10 sessions, c_only mode, paired on the same learners and seeds. Reward is per decision from the 4th task on, with the main weights and tau 0.70.

In [3]:
class RuleScored(AgentPolicy):
    def __call__(self, obs, allowed, env): return env.rule_action()

EV0 = 9_300_000
EVAL = [sample_learner(EV0 + i) for i in range(100)]
def evaluate(make):
    rows = []
    for i, L in enumerate(EVAL):
        p = make(i)
        d, s = run_learner(p, L, n_sessions=10, seed=EV0 + i, sim=SIM, session_mode="c_only")
        a = d[d.tasks_done >= 3]
        rows.append(dict(reward=np.mean(p.rewards), first_try=s.first_try_rate.mean(), safety_events=s.safety_override_count.mean(),
                         dropout=s.dropout.mean(), mean_difficulty=s.mean_difficulty.mean(), share_at_level_2=(a.difficulty == 2).mean(),
                         share_hold=(a.action == "hold").mean(), share_increase=(a.action == "increase_difficulty").mean(),
                         share_decrease=(a.action == "decrease_difficulty").mean()))
    return pd.DataFrame(rows)
Q_ev = evaluate(lambda i: AgentPolicy(Q_C.greedy(), TAU, WEIGHTS, seed=i, learn=False))
R_ev = evaluate(lambda i: RuleScored(QAgent(), TAU, WEIGHTS, seed=i, learn=False))
cols = ["reward", "first_try", "safety_events", "dropout", "mean_difficulty"]
QC = pd.DataFrame({"rule": R_ev[cols].apply(pm), "greedy Q (c_only)": Q_ev[cols].apply(pm)}).T
dif = Q_ev[cols] - R_ev[cols]
QC.loc["Q - rule (paired)"] = [pdiff(dif[c]) if dif[c].std() > 0 else "+0.0000 ± 0.0000" for c in cols]
display(QC); save_table(QC.reset_index().rename(columns={"index": "policy"}), "c_only_quick_check")
room = pd.DataFrame({"rule": R_ev[["share_at_level_2", "share_decrease", "share_hold", "share_increase"]].mean(),
                     "greedy Q": Q_ev[["share_at_level_2", "share_decrease", "share_hold", "share_increase"]].mean()}).T.round(3)
print("room left by the c_only cap (levels 1-2 only): share of decisions made at level 2, and the proposed-action mix"); display(room); save_table(room.reset_index(), "c_only_room")

,reward,first_try,safety_events,dropout,mean_difficulty
rule,0.5101 ± 0.0030,0.6972 ± 0.0037,0.0730 ± 0.0185,0.0000 ± 0.0000,1.7439 ± 0.0076
greedy Q (c_only),0.5226 ± 0.0022,0.7318 ± 0.0040,0.0350 ± 0.0123,0.0000 ± 0.0000,1.4020 ± 0.0014
Q - rule (paired),+0.0125 ± 0.0018,+0.0345 ± 0.0018,-0.0380 ± 0.0131,+0.0000 ± 0.0000,-0.3419 ± 0.0072


room left by the c_only cap (levels 1-2 only): share of decisions made at level 2, and the proposed-action mix


,share_at_level_2,share_decrease,share_hold,share_increase
rule,0.819,0.092,0.447,0.461
greedy Q,0.482,0.471,0.043,0.487


### Findings — step 2

100 new learners (seeds 9,300,000+), 10 sessions, c_only, mean ± SE, paired on the same learners and seeds:

| | reward | first-try | safety events / session | dropout | mean difficulty |
|---|---|---|---|---|---|
| rule | 0.5101 ± 0.0030 | 0.697 | 0.073 | 0 | 1.74 |
| greedy Q (c_only) | 0.5226 ± 0.0022 | 0.732 | 0.035 | 0 | 1.40 |
| Q − rule (paired) | +0.0125 ± 0.0018 | +0.0345 ± 0.0018 | −0.038 ± 0.013 | 0 | −0.34 ± 0.007 |

- Q beats the rule on reward (+0.0125, about 7 SE), first-try success and safety events, but **the room is small**: c_only offers only levels 1 and 2, there is no mastery to grow and no boredom dropout (dropout is 0 for every policy by construction), so the only things at stake are comfort-type metrics. Reward moves by 0.0125, not the +0.040 seen in combined mode.
- Behaviour: the rule spends 82 % of its decisions at level 2, the Q table only 48 %. Q proposes decrease / hold / increase 47 / 4 / 49 % of the time (rule 9 / 45 / 46 %), i.e. it moves between the two levels instead of holding. Because there is no learning signal in c_only, this should be read as a safe-and-comfortable policy, not a learning policy.

## 3. Export the live tables and golden vectors

Canonical JSON (sorted keys, compact separators, UTF-8), written byte-exactly so the sha256 in `MANIFEST.json` is the hash of the file as stored. State key = `"difficulty,first_try_bucket,strain_bucket,mastery_bucket"`
(mastery bucket 3 = unknown). A state missing from the table has all-zero Q-values (the agent's own behaviour). The golden vectors (200 per table) are produced with the `QAgent` of this notebook, greedy, and must be reproduced exactly by the backend.

In [4]:
OUT = cfg.RL_ROOT.parent / "backend" / "provenance" / "c_policy"
OUT.mkdir(parents=True, exist_ok=True)
STATUS = "APPROVED_FOR_PROVISIONAL_IMPLEMENTATION"     # same status the other code-gated families carry in the live app
canon = lambda obj: json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
key = lambda s: ",".join(map(str, s))

def artifact(name, ag, mode, n_train, seed):
    return {"artifact_id": f"c-rl-q-v1/{name}", "policy_version": "c-rl-q-v1", "session_mode": mode, "parameter_source_status": STATUS,
            "actions": list(E.ACTIONS), "state_key_format": "difficulty,first_try_bucket,strain_bucket,mastery_bucket",
            "bucket_edges": {"first_try": [0.4, 0.7], "strain": [0.3, 0.6], "mastery": [0.33, 0.66], "mastery_unknown_bucket": UNKNOWN_MASTERY,
                             "rule": "bucket = 0 if x < lo else 1 if x < hi else 2"},
            "tau": TAU, "weights": WEIGHTS, "min_tasks": ag.min_tasks, "tie_break_order": list(E.ACTIONS),
            "training": {"n_train_learners": n_train, "sessions_per_learner": SESSIONS, "seed": seed, "epsilon": 0.10, "alpha": 0.10, "gamma": 0.90,
                         "sim": dict(growth=True, boredom=True, eta=E.ETA, drop_a=E.DROP_A, drop_b=E.DROP_B, drop_c=E.DROP_C)},
            "q": {key(s): [float(x) for x in q] for s, q in sorted(ag.Q.items())},
            "visits": {key(s): int(n) for s, n in sorted(ag.visits.items())},
            "disclosure": cfg.DISCLOSURE}

meta_main = json.loads((cfg.MODELS_DIR / "q_main.json").read_text(encoding="utf-8"))
ARTS = {"q_c_only": (artifact("q_c_only", Q_C, "c_only", N_TRAIN, TRAIN_SEED0), Q_C),
        "q_main": (artifact("q_main", Q_M, "combined", meta_main["n_train_learners"], meta_main["seed"]), Q_M)}
manifest = {"schema": "c-policy-manifest-1", "policy_version": "c-rl-q-v1", "disclosure": cfg.DISCLOSURE, "artifacts": {}}
for name, (art, ag) in ARTS.items():
    raw = canon(art); (OUT / f"{name}.json").write_bytes(raw)
    manifest["artifacts"][name] = {"file": f"{name}.json", "sha256": hashlib.sha256(raw).hexdigest(), "session_mode": art["session_mode"],
                                   "parameter_source_status": STATUS, "n_states": len(art["q"]), "bytes": len(raw)}

# golden vectors
rng = np.random.default_rng(20261020)
EDGE = {"first_try": [0.0, 0.39999, 0.4, 0.69999, 0.7, 1.0], "strain": [0.0, 0.29999, 0.3, 0.59999, 0.6, 1.0], "mastery": [0.0, 0.32999, 0.33, 0.65999, 0.66, 1.0]}
CENTER = {"first_try": [0.2, 0.55, 0.85], "strain": [0.15, 0.45, 0.8], "mastery": [0.15, 0.5, 0.85]}
def vectors(name, ag, mode, n=200):
    visited = sorted(ag.visits)
    rows = []
    for i in range(n):
        if i % 2 == 0:     # a visited state, via bucket-centre values
            s = visited[int(rng.integers(len(visited)))]
            o = {"difficulty": s[0], "first_try": CENTER["first_try"][s[1]], "strain": CENTER["strain"][s[2]],
                 "mastery": None if s[3] == UNKNOWN_MASTERY else CENTER["mastery"][s[3]]}
        else:              # anything, including bucket edges
            pick = lambda k: float(rng.choice(EDGE[k])) if rng.random() < .5 else float(round(rng.random(), 5))
            o = {"difficulty": int(rng.integers(1, 6)), "first_try": pick("first_try"), "strain": pick("strain"),
                 "mastery": None if mode == "c_only" else pick("mastery")}
        o["tasks_done"] = int(rng.integers(0, 9))
        allowed = [a for a in E.ACTIONS if rng.random() < .6] or [str(rng.choice(E.ACTIONS))]
        st = state_of(o)
        act = ag.greedy().act(o, allowed, rng)
        rows.append({"obs": o, "state_key": key(st), "allowed": allowed, "tasks_done": o["tasks_done"], "expected_action": act,
                     "q_values": [float(x) for x in ag.Q.get(st, np.zeros(3))]})
    return rows
GOLD = {"schema": "c-policy-golden-vectors-1", "policy_version": "c-rl-q-v1", "disclosure": cfg.DISCLOSURE,
        "vectors": {name: vectors(name, ag, art["session_mode"]) for name, (art, ag) in ARTS.items()}}
graw = canon(GOLD); (OUT / "golden_vectors.json").write_bytes(graw)
manifest["golden_vectors"] = {"file": "golden_vectors.json", "sha256": hashlib.sha256(graw).hexdigest(), "rows_per_artifact": 200}
(OUT / "MANIFEST.json").write_bytes(canon(manifest))

for name in ARTS:
    assert hashlib.sha256((OUT / f"{name}.json").read_bytes()).hexdigest() == manifest["artifacts"][name]["sha256"]
sumry = pd.DataFrame([{"file": f, "sha256": m["sha256"], "bytes": m["bytes"], "states": m["n_states"]} for f, m in
                      ((k, v) for k, v in manifest["artifacts"].items())])
display(sumry); save_table(sumry, "export_manifest")
for name, rows in GOLD["vectors"].items():
    h = [r for r in rows if r["tasks_done"] < 3]
    print(f"{name}: {len(rows)} golden rows; {len(h)} with tasks_done < 3 (forced hold); actions {pd.Series([r['expected_action'] for r in rows]).value_counts().to_dict()}; "
          f"{sum(r['state_key'] not in ARTS[name][0]['q'] for r in rows)} rows in unvisited states (all-zero Q)")
print("golden_vectors sha256:", manifest["golden_vectors"]["sha256"])
print(f"notebook run time: {(time.time() - T_NB) / 60:.1f} min")

,file,sha256,bytes,states
0,q_c_only,d416772078679f3ebe2a41b554f59cdf8e42918fdc0b1a...,2200,18
1,q_main,65663066ba3552891b58cbfff6beb692f719b6412c5458...,9748,122


q_c_only: 200 golden rows; 69 with tasks_done < 3 (forced hold); actions {'hold': 112, 'decrease_difficulty': 57, 'increase_difficulty': 31}; 70 rows in unvisited states (all-zero Q)
q_main: 200 golden rows; 56 with tasks_done < 3 (forced hold); actions {'hold': 102, 'decrease_difficulty': 61, 'increase_difficulty': 37}; 7 rows in unvisited states (all-zero Q)
golden_vectors sha256: 7ef711e877da136fdfd7fa243bd2cea15a48bab4d5013af52d597910cff515d4
notebook run time: 16.2 min


### Findings — step 3

- **Exported** to `backend/provenance/c_policy/` as canonical JSON (sorted keys, compact): `q_c_only.json` (2,200 bytes, 18 states, sha256 `d416772078679f3ebe2a41b554f59cdf8e42918fdc0b1a…`), `q_main.json` (9,748 bytes, 122 states, sha256 `65663066ba3552891b58cbfff6beb692f719b6412c5458…`), `golden_vectors.json` (200 rows per table, sha256 `7ef711e877da136fdfd7fa243bd2cea15a48bab4d5013af52d597910cff515d4`) and `MANIFEST.json` (the full hashes; reload asserted equal to the stored bytes).
- Each table carries the state-bucket edges, tau (0.70), weights, min_tasks (3), tie-break order, training settings and the disclosure line. A state missing from a table means all-zero Q-values.
- Golden vectors: half from visited states (bucket-centre values), half random including bucket edges; 69 / 56 rows have tasks_done < 3 (forced hold), 70 / 7 rows fall in states with all-zero Q-values (tie → decrease first). Note: in those forced-hold rows `hold` is the expected action even if `hold` is not in the row's allowed set — as in training; safety then disposes.

---
_Literature-informed synthetic data — not clinical evidence. Simulation-only; not approved for use with children._

## 6. LIVE-02 c_only fix

**Note on steps 1–3.** They were executed **before** the LIVE-02 simulator change below (no boredom in c_only) and describe `q_c_only.json` (v1). Their saved outputs are kept; re-running them now
would train against the changed simulator and would not reproduce those v1 numbers. This section is self-contained.

**Problem.** The live policy bounced 1→2→1→2 even for a child who succeeds: state (1,2,0,3) → increase, state (2,2,0,3) → decrease. Hypothesis: in c_only training there was no boredom effect, so staying at a
level the child succeeds at was never rewarded. **Fix (simulator, not policy):** boredom dropout is now tested in c_only too, on the learner's **latent** mastery of the task's skill (`d_norm <= m - 0.15`, same margin;
the policy still sees mastery = None → bucket "unknown"). DROP_A/B/C, the reward weights, tau and the state buckets are unchanged; mastery growth stays off in c_only (its P(correct) does not read mastery).

In [1]:
import sys, json, time, hashlib
from pathlib import Path
T6 = time.time()
RL_ROOT = Path.cwd().resolve().parent
for p_ in (RL_ROOT, RL_ROOT / "notebooks"):
    if str(p_) not in sys.path:
        sys.path.insert(0, str(p_))
import rlconfig as cfg
import rl_core
from rl_core import env as E
from rl_core.env import SimConfig, run_learner, sample_learner, rule_policy
from rl_core.agent import QAgent, AgentPolicy, state_of, WEIGHTS
import numpy as np, pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)
SIM = SimConfig(growth=True, boredom=True, eta=E.ETA)
TAU = 0.70
pm = lambda x: f"{x.mean():.4f} ± {x.std() / np.sqrt(x.count()):.4f}"
pdiff = lambda x: f"{x.mean():+.4f} ± {x.std() / np.sqrt(x.count()):.4f}"
save_table = lambda df, n: df.to_csv(cfg.TABLES_DIR / f"04_{n}.csv", index=False)

# 1. diagnosis on v1
Q1 = QAgent.load(cfg.MODELS_DIR / "q_c_only.json")
print("v1 (q_c_only.json), q-values [decrease, hold, increase], visits:")
for s in [(1, 2, 0, 3), (2, 2, 0, 3)]:
    print(" ", s, np.round(Q1.Q[s], 4), Q1.visits[s])
q = Q1.Q[(2, 2, 0, 3)]
print(f"at (2,2,0,3): decrease - hold = {q[0] - q[1]:+.4f}  (decrease {q[0]:.4f}, hold {q[1]:.4f}, increase {q[2]:.4f}); greedy = {Q1.greedy()._best((2, 2, 0, 3), list(E.ACTIONS))}")

v1 (q_c_only.json), q-values [decrease, hold, increase], visits:
  (1, 2, 0, 3) [0.     2.3064 3.1256] 251107
  (2, 2, 0, 3) [2.7136 2.0763 0.    ] 179054
at (2,2,0,3): decrease - hold = +0.6372  (decrease 2.7136, hold 2.0763, increase 0.0000); greedy = decrease_difficulty


### Findings — 6.1 diagnosis

q_c_only.json (v1), Q-values [decrease, hold, increase] and visits:

| state | meaning | Q | visits |
|---|---|---|---|
| (1,2,0,3) | level 1, first-try high, strain low, mastery unknown | [0, 2.3064, 3.1256] | 251,107 |
| (2,2,0,3) | level 2, first-try high, strain low, mastery unknown | [2.7136, 2.0763, 0] | 179,054 |

At (2,2,0,3) **decrease beats hold by 0.637** (2.714 vs 2.076), so the greedy policy steps a succeeding child back down; at (1,2,0,3) increase beats hold by 0.82, so it steps up again — the 1→2→1→2 bounce. Both cells are heavily visited, so this is a converged value, not noise. Consistent with the hypothesis (not proven by this table alone): the c_only simulator had no cost for staying at a level the child already masters, so only the reward terms steer the choice.

In [2]:
# 2. rule policy, c_only, with the new boredom dropout (same cohort as NB02 step 2: 100 learners x 10 sessions)
print("DROP_A, DROP_B, DROP_C =", E.DROP_A, E.DROP_B, E.DROP_C)
LRN = [sample_learner(100 + i) for i in range(100)]
dr, ts = [], []
for i, L in enumerate(LRN):
    _, s = run_learner(rule_policy, L, 10, seed=100 + i, sim=SIM, session_mode="c_only")
    dr.append(s.dropout.mean()); ts.append(s.tasks.mean())
print(f"rule policy, c_only, boredom on: early-end rate {np.mean(dr):.3f}; tasks per session {np.mean(ts):.2f}  (combined, NB02 step 2: 0.145)")

DROP_A, DROP_B, DROP_C = -5.2 0.6 1.0


rule policy, c_only, boredom on: early-end rate 0.148; tasks per session 11.22  (combined, NB02 step 2: 0.145)


### Findings — 6.2 simulator change

- `rl_core/env.py`: boredom dropout is now tested in c_only too, using the learner's **latent** mastery of the task's skill (`d_norm <= m − 0.15`, same margin); in combined mode the generator's session-start mastery is used as before. The policy still sees `mastery = None` (bucket "unknown"). DROP_A/B/C = −5.2 / 0.6 / 1.0 unchanged. Mastery growth stays off in c_only (the c_only generator's P(correct) does not read mastery — `fallback_mode_p_correct` — so growth could not change any c_only outcome; stated in a code comment).
- **Rule policy, c_only, 100 learners × 10 sessions: early-end rate 14.8 %** (11.22 tasks per session), against 14.5 % in combined mode (NB02 step 2) — the same calibration carries over.
- **Combined mode unchanged: yes.** `tests/test_env.py::test_combined_trajectory_is_unchanged_by_the_c_only_boredom_change` pins a SHA-256 of a 6-session combined trajectory (72 decisions) taken before the change; a second test checks that c_only boredom dropout now exists while the observation's mastery is always None. Component C RL tests: 23 passed.

In [3]:
# 3. retrain c_only Q v2: NB04 settings (3,000 learners x 25 sessions), a NEW training seed base (< 9,000,000)
V2_SEED0, N_TRAIN, SESSIONS = 50_000, 3000, 25
agent2 = QAgent(); pol2 = AgentPolicy(agent2, TAU, WEIGHTS, seed=V2_SEED0)
t0, br = time.time(), []
for i in range(N_TRAIN):
    pol2.rewards.clear()
    run_learner(pol2, sample_learner(V2_SEED0 + i), n_sessions=SESSIONS, seed=V2_SEED0 + i, sim=SIM, session_mode="c_only")
    br.append((np.sum(pol2.rewards), len(pol2.rewards)))
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{N_TRAIN}  {time.time() - t0:.0f}s  states visited {len(agent2.visits)}", flush=True)
B = np.array(br)
print(f"v2 trained: {(time.time() - t0) / 60:.1f} min, {int(B[:, 1].sum()):,} decisions; reward first 500 learners {B[:500, 0].sum() / B[:500, 1].sum():.4f}, last 500 {B[-500:, 0].sum() / B[-500:, 1].sum():.4f}")
agent2.save(cfg.MODELS_DIR / "q_c_only_v2.json", tau=TAU, weights=WEIGHTS, n_train_learners=N_TRAIN, sessions_per_learner=SESSIONS, seed=V2_SEED0,
            session_mode="c_only", artifact_version="v2", disclosure=cfg.DISCLOSURE)
Q2 = QAgent.load(cfg.MODELS_DIR / "q_c_only_v2.json")
g2 = Q2.greedy()
print("v2 q-values [decrease, hold, increase], visits:")
for s in [(1, 2, 0, 3), (2, 2, 0, 3)]:
    print(" ", s, np.round(Q2.Q[s], 4), Q2.visits[s], "-> greedy", g2._best(s, list(E.ACTIONS)))

  500/3000  150s  states visited 18


  1000/3000  341s  states visited 18


  1500/3000  489s  states visited 18


  2000/3000  650s  states visited 18


  2500/3000  797s  states visited 18


  3000/3000  957s  states visited 18


v2 trained: 16.0 min, 666,400 decisions; reward first 500 learners 0.5057, last 500 0.5062
v2 q-values [decrease, hold, increase], visits:
  (1, 2, 0, 3) [0.     2.4962 2.9084] 175494 -> greedy increase_difficulty
  (2, 2, 0, 3) [2.307  2.7117 0.    ] 192316 -> greedy hold


### Findings — 6.3 retrain

- c_only Q v2: 3,000 learners × 25 sessions, training seeds 50,000+ (new, < 9,000,000), 16.0 min, 666,400 decisions, 18 states visited (as in v1). Reward per decision is flat (first 500 learners 0.5057, last 500 0.5062).
- Saved as `outputs/models/q_c_only_v2.json`. Q-values [decrease, hold, increase]: **(2,2,0,3) = [2.307, 2.712, 0] → greedy HOLD** (v1: decrease); (1,2,0,3) = [0, 2.496, 2.908] → greedy increase. The decrease-vs-hold gap at (2,2,0,3) went from +0.637 to −0.405.

In [4]:
# 4. evaluation: 100 new c_only learners x 10 sessions (seeds 9,400,000+, new), rule vs Q v1 vs Q v2, same learners and seeds
class RuleScored(AgentPolicy):
    def __call__(self, obs, allowed, env): return env.rule_action()

EV0 = 9_400_000
EVAL = [sample_learner(EV0 + i) for i in range(100)]
MOVE = {"increase_difficulty": 1, "decrease_difficulty": -1, "hold": 0}
def evaluate(make):
    rows = []
    for i, L in enumerate(EVAL):
        p_ = make(i)
        d, s = run_learner(p_, L, n_sessions=10, seed=EV0 + i, sim=SIM, session_mode="c_only")
        mv = d.applied_action.map(MOVE).values
        same_session = (d.session.values[1:] == d.session.values[:-1])
        flips = same_session & (mv[1:] * mv[:-1] == -1)                   # a move that reverses the previous decision's move
        moves = same_session & (mv[1:] != 0)
        hi = (d.first_try >= 0.7) & (d.strain < 0.3) & (d.tasks_done >= 3)
        rows.append(dict(reward=np.mean(p_.rewards), first_try=s.first_try_rate.mean(), safety_events=s.safety_override_count.mean(),
                         dropout=s.dropout.mean(), mean_difficulty=s.mean_difficulty.mean(),
                         flips=int(flips.sum()), decisions=int(same_session.sum()), moves=int(moves.sum()),
                         l2_share_hi_ft_low_strain=(d.difficulty[hi] == 2).mean() if hi.any() else np.nan, n_hi=int(hi.sum())))
    return pd.DataFrame(rows)
RES = {"rule": evaluate(lambda i: RuleScored(QAgent(), TAU, WEIGHTS, seed=i, learn=False)),
       "Q v1": evaluate(lambda i: AgentPolicy(Q1.greedy(), TAU, WEIGHTS, seed=i, learn=False)),
       "Q v2": evaluate(lambda i: AgentPolicy(Q2.greedy(), TAU, WEIGHTS, seed=i, learn=False))}
cols = ["reward", "first_try", "safety_events", "dropout", "mean_difficulty", "l2_share_hi_ft_low_strain"]
T6T = pd.DataFrame({k: v[cols].apply(pm) for k, v in RES.items()}).T
T6T["flip rate (flips / decisions)"] = [f"{v.flips.sum() / v.decisions.sum():.4f}" for v in RES.values()]
T6T["flips / moves"] = [f"{v.flips.sum() / v.moves.sum():.4f}" for v in RES.values()]
T6T["n decisions at L2-ish cell"] = [int(v.n_hi.sum()) for v in RES.values()]
display(T6T); save_table(T6T.reset_index().rename(columns={"index": "policy"}), "live02_eval")
for a, b in (("Q v2", "Q v1"), ("Q v2", "rule")):
    d_ = RES[a][["reward", "first_try", "safety_events", "dropout", "mean_difficulty"]] - RES[b][["reward", "first_try", "safety_events", "dropout", "mean_difficulty"]]
    print(f"paired {a} - {b}:", {c: pdiff(d_[c]) for c in d_.columns})
print("Q v2 greedy action at (2,2,0,3):", g2._best((2, 2, 0, 3), list(E.ACTIONS)), np.round(Q2.Q[(2, 2, 0, 3)], 4))
print("Q v1 greedy action at (2,2,0,3):", Q1.greedy()._best((2, 2, 0, 3), list(E.ACTIONS)))
f1 = RES["Q v1"].flips.sum() / RES["Q v1"].decisions.sum(); f2 = RES["Q v2"].flips.sum() / RES["Q v2"].decisions.sum()
print(f"GATE inputs: v2 greedy at (2,2,0,3) is decrease: {g2._best((2, 2, 0, 3), list(E.ACTIONS)) == 'decrease_difficulty'}; flip rate v1 {f1:.4f} -> v2 {f2:.4f}")
print(f"section 6 run time: {(time.time() - T6) / 60:.1f} min")

,reward,first_try,safety_events,dropout,mean_difficulty,l2_share_hi_ft_low_strain,flip rate (flips / decisions),flips / moves,n decisions at L2-ish cell
rule,0.4973 ± 0.0031,0.6944 ± 0.0047,0.0510 ± 0.0146,0.1440 ± 0.0123,1.7197 ± 0.0090,0.8920 ± 0.0071,0.0029,0.0535,4441
Q v1,0.5071 ± 0.0028,0.7332 ± 0.0050,0.0380 ± 0.0126,0.1730 ± 0.0127,1.3695 ± 0.0030,0.4905 ± 0.0032,0.7466,0.8734,5008
Q v2,0.5014 ± 0.0032,0.7139 ± 0.0050,0.0480 ± 0.0142,0.1570 ± 0.0122,1.5513 ± 0.0056,0.8281 ± 0.0054,0.1621,0.4739,4645


paired Q v2 - Q v1: {'reward': '-0.0058 ± 0.0018', 'first_try': '-0.0193 ± 0.0018', 'safety_events': '+0.0100 ± 0.0100', 'dropout': '-0.0160 ± 0.0044', 'mean_difficulty': '+0.1818 ± 0.0037'}
paired Q v2 - rule: {'reward': '+0.0041 ± 0.0015', 'first_try': '+0.0194 ± 0.0021', 'safety_events': '-0.0030 ± 0.0041', 'dropout': '+0.0130 ± 0.0075', 'mean_difficulty': '-0.1684 ± 0.0088'}
Q v2 greedy action at (2,2,0,3): hold [2.307  2.7117 0.    ]
Q v1 greedy action at (2,2,0,3): decrease_difficulty
GATE inputs: v2 greedy at (2,2,0,3) is decrease: False; flip rate v1 0.7466 -> v2 0.1621
section 6 run time: 16.8 min


### Findings — 6.4 evaluation and gate

100 new c_only learners × 10 sessions (seeds 9,400,000+), same learners and seeds, the new simulator (boredom on), mean ± SE:

| | reward | first-try | safety events | dropout | mean difficulty | flip rate | L2 share (first-try high, strain low) |
|---|---|---|---|---|---|---|---|
| rule | 0.4973 ± 0.0031 | 0.694 | 0.051 | 0.144 | 1.72 | 0.003 | 0.892 |
| Q v1 | 0.5071 ± 0.0028 | 0.733 | 0.038 | 0.173 | 1.37 | **0.747** | 0.491 |
| Q v2 | 0.5014 ± 0.0032 | 0.714 | 0.048 | 0.157 | 1.55 | **0.162** | **0.828** |

Flip rate = share of decisions (with a predecessor in the same session) whose applied move reverses the previous decision's move; flips / moves: rule 0.054, v1 0.873, v2 0.474.
Paired Q v2 − Q v1: reward −0.0058 ± 0.0018, first-try −0.019, dropout −0.016 ± 0.004, mean difficulty +0.18. Paired Q v2 − rule: reward +0.0041 ± 0.0015, first-try +0.019, dropout +0.013 ± 0.008, mean difficulty −0.17.

- **Q v2 greedy at (2,2,0,3): hold** (v1: decrease_difficulty).
- **Gate: PASS** — v2 no longer decreases at (2,2,0,3), and the flip rate fell from 0.747 to 0.162 (4.6× lower).
- Honest caveats: v2 is **not** oscillation-free (flip rate 0.162 vs the rule's 0.003; nearly half of its moves still reverse the previous one), it spends less time at level 2 than the rule in the success cell (0.83 vs 0.89), and its reward is slightly below v1's (−0.006; plausibly because v1 sits at the easier level where first-try success is higher, which the reward favours — not tested). Against the rule it keeps small gains in reward and first-try success; dropout is 0.013 higher (≈ 1.7 SE). No other tuning was done.

### 6.5 Ship v2 (gate passed)

Export `q_c_only_v2.json` to `backend/provenance/c_policy/` (canonical JSON). `MANIFEST.json`: `q_c_only` now points to the v2 file (`artifact_version` "v2"); the v1 entry is kept as `q_c_only_v1`, marked
`superseded_by: q_c_only_v2.json`, and the v1 file stays on disk for the audit trail; the combined entry is untouched. The c_only golden vectors are regenerated (new seed) from the v2 table; the combined rows are kept
byte-identical.

In [1]:
import sys, json, hashlib
from pathlib import Path
RL_ROOT = Path.cwd().resolve().parent
for p_ in (RL_ROOT, RL_ROOT / "notebooks"):
    if str(p_) not in sys.path:
        sys.path.insert(0, str(p_))
import rlconfig as cfg
from rl_core import env as E
from rl_core.agent import QAgent, state_of, WEIGHTS, UNKNOWN_MASTERY
import numpy as np, pandas as pd
OUT = cfg.RL_ROOT.parent / "backend" / "provenance" / "c_policy"
TAU, SESSIONS, STATUS = 0.70, 25, "APPROVED_FOR_PROVISIONAL_IMPLEMENTATION"
canon = lambda obj: json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
key = lambda s: ",".join(map(str, s))
Q2 = QAgent.load(cfg.MODELS_DIR / "q_c_only_v2.json")
meta2 = json.loads((cfg.MODELS_DIR / "q_c_only_v2.json").read_text(encoding="utf-8"))

art = {"artifact_id": "c-rl-q-v1/q_c_only_v2", "artifact_version": "v2", "policy_version": "c-rl-q-v1", "session_mode": "c_only",
       "parameter_source_status": STATUS, "actions": list(E.ACTIONS),
       "state_key_format": "difficulty,first_try_bucket,strain_bucket,mastery_bucket",
       "bucket_edges": {"first_try": [0.4, 0.7], "strain": [0.3, 0.6], "mastery": [0.33, 0.66], "mastery_unknown_bucket": UNKNOWN_MASTERY,
                        "rule": "bucket = 0 if x < lo else 1 if x < hi else 2"},
       "tau": TAU, "weights": WEIGHTS, "min_tasks": Q2.min_tasks, "tie_break_order": list(E.ACTIONS),
       "training": {"n_train_learners": meta2["n_train_learners"], "sessions_per_learner": SESSIONS, "seed": meta2["seed"], "epsilon": 0.10, "alpha": 0.10, "gamma": 0.90,
                    "sim": dict(growth=True, boredom=True, eta=E.ETA, drop_a=E.DROP_A, drop_b=E.DROP_B, drop_c=E.DROP_C),
                    "note": "LIVE-02: c_only trained with boredom dropout on the latent mastery (policy still sees mastery unknown)"},
       "q": {key(s): [float(x) for x in q] for s, q in sorted(Q2.Q.items())},
       "visits": {key(s): int(n) for s, n in sorted(Q2.visits.items())},
       "disclosure": cfg.DISCLOSURE}
raw = canon(art); (OUT / "q_c_only_v2.json").write_bytes(raw)
sha2 = hashlib.sha256(raw).hexdigest()

old = json.loads((OUT / "MANIFEST.json").read_bytes())
assert "q_c_only_v1" not in old["artifacts"], "already shipped"
v1 = dict(old["artifacts"]["q_c_only"]); v1["superseded_by"] = "q_c_only_v2.json"
assert hashlib.sha256((OUT / v1["file"]).read_bytes()).hexdigest() == v1["sha256"]          # the v1 file is intact
manifest = {**old, "artifacts": {"q_c_only": {"file": "q_c_only_v2.json", "sha256": sha2, "session_mode": "c_only", "artifact_version": "v2",
                                              "parameter_source_status": STATUS, "n_states": len(art["q"]), "bytes": len(raw)},
                                 "q_c_only_v1": v1, "q_main": old["artifacts"]["q_main"]}}

# golden vectors: new c_only rows from v2; combined rows kept byte-identical
rng = np.random.default_rng(20261021)
EDGE = {"first_try": [0.0, 0.39999, 0.4, 0.69999, 0.7, 1.0], "strain": [0.0, 0.29999, 0.3, 0.59999, 0.6, 1.0]}
CENTER = {"first_try": [0.2, 0.55, 0.85], "strain": [0.15, 0.45, 0.8]}
visited = sorted(Q2.visits); rows = []
for i in range(200):
    if i % 2 == 0:
        s = visited[int(rng.integers(len(visited)))]
        o = {"difficulty": s[0], "first_try": CENTER["first_try"][s[1]], "strain": CENTER["strain"][s[2]], "mastery": None}
    else:
        pick = lambda k: float(rng.choice(EDGE[k])) if rng.random() < .5 else float(round(rng.random(), 5))
        o = {"difficulty": int(rng.integers(1, 6)), "first_try": pick("first_try"), "strain": pick("strain"), "mastery": None}
    o["tasks_done"] = int(rng.integers(0, 9))
    allowed = [a for a in E.ACTIONS if rng.random() < .6] or [str(rng.choice(E.ACTIONS))]
    st = state_of(o)
    rows.append({"obs": o, "state_key": key(st), "allowed": allowed, "tasks_done": o["tasks_done"],
                 "expected_action": Q2.greedy().act(o, allowed, rng), "q_values": [float(x) for x in Q2.Q.get(st, np.zeros(3))]})
gold_old = json.loads((OUT / "golden_vectors.json").read_bytes())
gold = {**gold_old, "vectors": {"q_c_only": rows, "q_main": gold_old["vectors"]["q_main"]}}
assert canon(gold_old["vectors"]["q_main"]) == canon(gold["vectors"]["q_main"])
graw = canon(gold); (OUT / "golden_vectors.json").write_bytes(graw)
manifest["golden_vectors"] = {**old["golden_vectors"], "sha256": hashlib.sha256(graw).hexdigest()}
(OUT / "MANIFEST.json").write_bytes(canon(manifest))

for k, m in manifest["artifacts"].items():
    assert hashlib.sha256((OUT / m["file"]).read_bytes()).hexdigest() == m["sha256"], k
print("q_c_only (v2):", sha2, len(raw), "bytes,", len(art["q"]), "states")
print("q_c_only_v1  :", v1["sha256"], "superseded_by", v1["superseded_by"])
print("q_main       :", manifest["artifacts"]["q_main"]["sha256"], "(unchanged:", manifest["artifacts"]["q_main"] == old["artifacts"]["q_main"], ")")
print("golden_vectors:", manifest["golden_vectors"]["sha256"], "| c_only rows:", pd.Series([r["expected_action"] for r in rows]).value_counts().to_dict(),
      "| rows at (2,2,0,3):", [(r["tasks_done"], r["expected_action"]) for r in rows if r["state_key"] == "2,2,0,3"][:6])

q_c_only (v2): 5893e91f8917cdabc72a5a9b737c93267f9a55b1b4acbea03353b491218f0e6b 2340 bytes, 18 states
q_c_only_v1  : d416772078679f3ebe2a41b554f59cdf8e42918fdc0b1a1bec5a1c9438420bde superseded_by q_c_only_v2.json
q_main       : 65663066ba3552891b58cbfff6beb692f719b6412c545838d9611ff3d1fee42e (unchanged: True )
golden_vectors: 79df790f5ca21e1712f13922786e046d782e193fe6fabf9a85ee5d817ccaff1b | c_only rows: {'hold': 115, 'decrease_difficulty': 44, 'increase_difficulty': 41} | rows at (2,2,0,3): [(1, 'hold'), (0, 'hold'), (4, 'hold'), (7, 'hold'), (5, 'decrease_difficulty')]


### Findings — 6.5 export

- `q_c_only_v2.json`: 2,340 bytes, 18 states, sha256 `5893e91f8917cdabc72a5a9b737c93267f9a55b1b4acbea03353b491218f0e6b` (canonical JSON, `artifact_version: "v2"`).
- `MANIFEST.json`: `q_c_only` → `q_c_only_v2.json` (v2); the v1 entry is kept as `q_c_only_v1` with `superseded_by: q_c_only_v2.json` (the v1 file `q_c_only.json`, sha256 `d416772078679f3e…`, is untouched and still verified); `q_main` entry byte-identical (`65663066ba3552891b58cbfff6beb692f719b6412c545838d9611ff3d1fee42e`).
- `golden_vectors.json` regenerated for c_only (200 rows from v2, new seed; 115 hold / 44 decrease / 41 increase); the combined rows are byte-identical to before (asserted). New golden sha256 `79df790f5ca21e1712f13922786e046d782e193fe6fabf9a85ee5d817ccaff1b`.

---
_Literature-informed synthetic data — not clinical evidence. Simulation-only; not approved for use with children._